# Analyse mechanism results

Read-only orchestrator over the result tables (Volt-VAr proxy, Volt-Watt proxy, response observability) via `result_views.py` and `result_plots.py`. 

In [ ]:
from __future__ import annotations

import dataclasses
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'src' / 'dnsp_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the dnsp_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dnsp_analysis.config import load_config
from dnsp_analysis.mechanism_config import load_mechanism_config
from dnsp_analysis import result_plots as rp
from dnsp_analysis import result_views as rv

CONFIG_PATH = PROJECT_ROOT / 'analysis.toml'
config = load_config(CONFIG_PATH, check_inputs=True)
mechanism_der_inferred = load_mechanism_config(CONFIG_PATH)
mechanism_all_phases = dataclasses.replace(mechanism_der_inferred, phase_scope_basis='all_phases').validate()
mechanism_solar_proxy = dataclasses.replace(mechanism_der_inferred, capacity_basis='solar_capacity_kw_proxy').validate()
mechanism_p99_proxy = dataclasses.replace(mechanism_der_inferred, capacity_basis='p99_net_export_proxy').validate()
TRACKS = {
    'der_inferred': mechanism_der_inferred,
    'all_phases': mechanism_all_phases,
    'solar_capacity_kw_proxy': mechanism_solar_proxy,
    'p99_net_export_proxy': mechanism_p99_proxy,
}
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')

## Stage 0: purpose, guardrails and source inventory

Lists every result file for both tracks (existence, size, row count, methodology id) and pulls the reconciled methodology/provenance context for each. 

`result_context` raises if a table's own methodology/provenance columns are internally mixed. 

The one expected cross-table difference `response_observability`'s methodology id not matching an `all_phases` curve-table run, because that table is never rebuilt per track. 
Is reported as an explicit flag rather than an error.

In [ ]:
SAMPLE_MONTH = '2025-04'
SAMPLE_SITE_BUCKET = 0
sample_scope = config.scope(SAMPLE_MONTH, SAMPLE_SITE_BUCKET)
full_scope = config.scope(None, None)

contexts = {}
for track_name, mechanism in TRACKS.items():
    print(f'--- {track_name} ---')
    inventory = rv.result_inventory(config, full_scope, mechanism=mechanism)
    display(inventory)
    context = rv.result_context(config, full_scope, mechanism=mechanism)
    contexts[track_name] = context
    display(pd.Series(context, name='value').to_frame())
    assert context['formal_inverter_conformance_assessable'] is False, (
        f'{track_name}: formal_inverter_conformance_assessable must stay False -- '
        'this notebook never claims formal conformance.'
    )
    existing = inventory.loc[inventory['exists'], 'distinct_methodology_ids']
    assert (existing <= 1).all(), (
        f'{track_name}: a result file contains more than one methodology_id -- stop and investigate.'
    )

print()
print('response_observability_methodology_matches_curve_tables per track:')
for track_name, context in contexts.items():
    print(f"  {track_name}: {context['response_observability_methodology_matches_curve_tables']}")

## Stage 1: deterministic-slice validation

Same April 2025 / bucket 0 sample scope Notebook 4 uses. Loads the sample
mechanism outputs Notebook 4 already built and validates every
`result_views.py` view reconciles back to its own source file before any
plotting happens.

In [ ]:
sample_validation = {}
for track_name, mechanism in TRACKS.items():
    result = rv.validate_result_views(config, sample_scope, mechanism=mechanism)
    sample_validation[track_name] = result
    headline = {k: v for k, v in result.items() if k not in {'failures', 'checks'}}
    display(pd.Series(headline, name='value').to_frame())
    if result['failures']:
        display(result['failures'])
    assert result['status'] == 'pass', f'{track_name}: sample view reconciliation failed -- see failures above.'
print('Deterministic-slice view reconciliation passed for both tracks.')

## Stage 2: Volt-VAr denominator coverage (deterministic slice)

Sequential denominator precedence: a source interval is classified as the
first of `ineligible_site`, `missing_input`, `not_activated`,
`sign_unverified`, `capacity_unavailable`, `below_minimum_active_power` that
applies, else `assessable`. Every fraction below is of `n_source_intervals`.

In [ ]:
fig, axes = plt.subplots(1, len(TRACKS), figsize=(7.5 * len(TRACKS), 4.5))
sample_voltvar_denominator = {}
for ax, (track_name, mechanism) in zip(axes, TRACKS.items()):
    frame = rv.voltvar_denominator_view(config, sample_scope, mechanism=mechanism)
    sample_voltvar_denominator[track_name] = frame
    rp.plot_denominator_breakdown(frame, mechanism_name='Volt-VAr', context=contexts[track_name], ax=ax)
    ax.set_title(f'{track_name}: ' + ax.get_title())
plt.tight_layout()
plt.show()

## Stage 3: Volt-VAr proxy statuses (deterministic slice)

Only meaningful if `n_assessable > 0`. With `s_rated_kva` unavailable for
every site today, `n_assessable` is expected to be 0 -- the panel below
shows that honestly rather than reporting a 0% conformance rate.

In [ ]:
fig, axes = plt.subplots(1, len(TRACKS), figsize=(7.5 * len(TRACKS), 4.5))
sample_voltvar_status = {}
for ax, (track_name, mechanism) in zip(axes, TRACKS.items()):
    frame = rv.voltvar_status_view(config, sample_scope, mechanism=mechanism, minimum_denominator=30)
    sample_voltvar_status[track_name] = frame
    rp.plot_status_breakdown(frame, mechanism_name='Volt-VAr', context=contexts[track_name], ax=ax)
    ax.set_title(f'{track_name}: ' + ax.get_title())
    n_assessable = int(frame['n_assessable'].iloc[0])
    print(f'{track_name}: n_assessable = {n_assessable:,}')
plt.tight_layout()
plt.show()

## Stage 4: Volt-Watt denominator coverage (deterministic slice)

`not_activated` (voltage at or below V1) and `not_exporting` (net export
`<= 0`) are different filters: a site can be exporting but below the
Volt-Watt activation voltage, or above it but net-importing at that instant.

In [ ]:
fig, axes = plt.subplots(1, len(TRACKS), figsize=(7.5 * len(TRACKS), 4.5))
sample_voltwatt_denominator = {}
for ax, (track_name, mechanism) in zip(axes, TRACKS.items()):
    frame = rv.voltwatt_denominator_view(config, sample_scope, mechanism=mechanism)
    sample_voltwatt_denominator[track_name] = frame
    rp.plot_denominator_breakdown(frame, mechanism_name='Volt-Watt', context=contexts[track_name], ax=ax)
    ax.set_title(f'{track_name}: ' + ax.get_title())
plt.tight_layout()
plt.show()

## Stage 5: Volt-Watt proxy statuses (deterministic slice)

`proxy_does_not_exceed_curve_ceiling` is never conformance evidence --
household load can suppress net export below the ceiling regardless of
inverter behaviour.

In [ ]:
fig, axes = plt.subplots(1, len(TRACKS), figsize=(7.5 * len(TRACKS), 4.5))
sample_voltwatt_status = {}
for ax, (track_name, mechanism) in zip(axes, TRACKS.items()):
    frame = rv.voltwatt_status_view(config, sample_scope, mechanism=mechanism)
    sample_voltwatt_status[track_name] = frame
    rp.plot_status_breakdown(frame, mechanism_name='Volt-Watt', context=contexts[track_name], ax=ax)
    ax.set_title(f'{track_name}: ' + ax.get_title())
    n_assessable = int(frame['n_assessable'].iloc[0])
    print(f'{track_name}: n_assessable = {n_assessable:,}')
plt.tight_layout()
plt.show()

## Stage 6: response observability (deterministic slice)

Volt-VAr and Volt-Watt observability are shown separately below; both
remain association/direction evidence only, never a causal or conformance
claim. `response_observability.parquet` is the single shared table -- it is
identical for both tracks (see the Stage 0 methodology check above), so it
is read once, using `mechanism_der_inferred` only for its review-state
labels.

In [ ]:
sample_observability_status = rv.observability_status_view(config, sample_scope, mechanism=mechanism_der_inferred)
display(sample_observability_status.T)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
rp.plot_observability_status(sample_observability_status, mechanism_name='Volt-VAr', context=contexts['der_inferred'], ax=axes[0])
rp.plot_observability_status(sample_observability_status, mechanism_name='Volt-Watt', context=contexts['der_inferred'], ax=axes[1])
plt.tight_layout()
plt.show()

sample_observability_metric = rv.observability_metric_view(
    config, sample_scope, mechanism=mechanism_der_inferred,
    minimum_excited_intervals=mechanism_der_inferred.minimum_response_intervals,
)
display(sample_observability_metric.T)

## Stage 7: sample gate

Re-asserts, for both tracks, that every view reconciles, dimensions are
valid, classifications never exceed their own denominator, and provenance is
internally consistent, before unlocking the full-results stages below.

In [ ]:
for track_name, mechanism in TRACKS.items():
    result = rv.validate_result_views(config, sample_scope, mechanism=mechanism)
    assert result['status'] == 'pass', f"{track_name}: sample gate failed -- {result['failures']}"
    context = rv.result_context(config, sample_scope, mechanism=mechanism)
    assert context['formal_inverter_conformance_assessable'] is False
for track_name, frame in sample_voltvar_status.items():
    classified = sum(int(frame[c].iloc[0]) for c in rv.VOLTVAR_STATUS_COLUMNS)
    assert classified == int(frame['n_assessable'].iloc[0]), f'{track_name}: Volt-VAr classification exceeds n_assessable'
for track_name, frame in sample_voltwatt_status.items():
    classified = sum(int(frame[c].iloc[0]) for c in rv.VOLTWATT_STATUS_COLUMNS)
    assert classified == int(frame['n_assessable'].iloc[0]), f'{track_name}: Volt-Watt classification exceeds n_assessable'
print('Sample gate passed for both tracks -- proceeding to full-results analysis is safe.')

## Stage 8: deliberate full-results analysis

The remaining stages read the full Notebook 4 outputs (not a sample). They
are still read-only -- no `build_*` call appears below -- but can be
computationally heavier, so they stay behind an explicit, deliberate opt-in
per the acceptance spec, mirroring Notebook 4's own full-run gate.

In [ ]:
FULL_ANALYSIS_CONFIRMATION = 'RUN FULL RESULTS ANALYSIS'  # Change to: RUN FULL RESULTS ANALYSIS
for _name in TRACKS:
    assert sample_validation[_name]['status'] == 'pass', f'{_name}: sample validation not passing'
assert FULL_ANALYSIS_CONFIRMATION == 'RUN FULL RESULTS ANALYSIS', (
    "Review Stages 0-7 above, then set FULL_ANALYSIS_CONFIRMATION = 'RUN FULL RESULTS ANALYSIS'."
)
print('Full-results analysis unlocked:', full_scope.label)

## Stage 9: fleet summary and Volt-VAr classification (full dataset)

Kept deliberately simple -- more detailed breakdowns (by month, cohort,
voltage bin) can be added back once this baseline is validated.

**Eligibility, confirmed:** every row in every track below is already
restricted to `analysis_cohort = 'solar_only'` and `controlled_load = 'No'`
sites (plus no battery, high-confidence phase mapping, full power
coverage) -- this is `core_site_gate_sql` in `mechanism_results.py`, applied
unconditionally to every track, not something this notebook filters again.

**Fleet KPI table:** sites/timestamps analysed, sites/timestamps that
crossed the mechanism's activation threshold ("required a response"), and
-- Volt-VAr only -- timestamps where a response was actually observed.
Volt-Watt has no analogous "had a response" figure: being below the
Volt-Watt ceiling is not evidence of curtailment (no counterfactual P
exists to compare against), so that column is `NaN` rather than fabricated.

**Volt-VAr classification:** `Q_impact = sign x (Q_kvar / Q_voltvar)`,
where `Q_voltvar` is the nearest edge of the tolerance-clamped required
band and `sign` is +1 when measured Q points the same direction as that
edge, else -1 (see `q_impact_nearest_edge_sql`). Every assessable interval
falls into exactly one of six buckets -- each count is a count of
timestamps (source intervals), not sites:

| Bucket | Meaning |
|---|---|
| Conformant | measured Q falls inside the tolerance-clamped required band itself |
| Adverse | Q_impact < -10% (wrong direction) |
| Inactive | -10% to +10% (no meaningful response) |
| Major deficit | +10% to +90% (some response, not enough) |
| Minor deviation | +90% to +110% (close to required, outside the exact band) |
| Major surplus | > +110% (more than required) |

**Conformance rollup (reviewed methodology):** a business-rule grouping on
top of the six buckets above, not a new measurement --
`non-conformance = Adverse + Inactive + Major deficit`,
`conformance = Conformant + Minor deviation + Major surplus`. Shown at both
the timestamp level (fraction of assessable intervals) and the site level
(is *this site's own majority* of assessable intervals conformant -- default
threshold 50%, see `voltvar_site_conformance_view`). A site with zero
assessable intervals is reported separately as `not_assessable`, never
folded into non-conformant.

In [ ]:
fleet_summary = pd.DataFrame(
    [
        rv.fleet_summary_view(config, full_scope, mechanism_name=mechanism_name, mechanism=mechanism)
        for mechanism_name in ('Volt-VAr', 'Volt-Watt')
        for track_name, mechanism in TRACKS.items()
    ]
)
fleet_summary.insert(1, 'track', [t for _ in ('Volt-VAr', 'Volt-Watt') for t in TRACKS])
fleet_summary = fleet_summary.set_index(['mechanism', 'track'])
display(fleet_summary)

In [ ]:
full_voltvar_status = {
    track_name: rv.voltvar_status_view(config, full_scope, mechanism=mechanism)
    for track_name, mechanism in TRACKS.items()
}
rp.plot_voltvar_classification_by_track(full_voltvar_status)
plt.tight_layout()
plt.show()

print('Raw timestamp counts per bucket (not fractions):')
display(rp.voltvar_classification_counts_by_track(full_voltvar_status))

### Conformance rollup -- timestamp level and site level

In [ ]:
conformance_timestamp_level = pd.DataFrame(
    {
        track_name: {
            'n_assessable': int(frame['n_assessable'].iloc[0]),
            'n_conformance': int(frame['n_conformance'].iloc[0]),
            'n_non_conformance': int(frame['n_non_conformance'].iloc[0]),
            'conformance_fraction': frame['conformance_fraction_of_assessable'].iloc[0],
        }
        for track_name, frame in full_voltvar_status.items()
    }
).T
display(conformance_timestamp_level)

In [ ]:
site_conformance = {
    track_name: rv.voltvar_site_conformance_view(config, full_scope, mechanism=mechanism)
    for track_name, mechanism in TRACKS.items()
}
site_conformance_counts = pd.DataFrame(
    {track_name: frame['site_status'].value_counts() for track_name, frame in site_conformance.items()}
).fillna(0).astype(int)
display(site_conformance_counts)

In [ ]:
sc = site_conformance['p99_net_export_proxy']
sc.to_csv('sites_breakdown_p99_net_export_proxy.csv')
sc

### Volt-Watt curve status by track

Volt-Watt has only two statuses -- exceeds the ceiling or does not.
`proxy_does_not_exceed_curve_ceiling` is never relabelled conformance:
household load can suppress net export below the ceiling regardless of
inverter behaviour, so there is no equivalent conformance rollup here.

In [ ]:
full_voltwatt_status = {
    track_name: rv.voltwatt_status_view(config, full_scope, mechanism=mechanism)
    for track_name, mechanism in TRACKS.items()
}
rp.plot_voltwatt_classification_by_track(full_voltwatt_status)
plt.tight_layout()
plt.show()

In [ ]:
full_observability_by_month = rv.observability_status_view(
    config, full_scope, mechanism=mechanism_der_inferred, dimensions=('year_utc', 'month_utc')
)
display(full_observability_by_month[['year_utc', 'month_utc', 'n_site_phase_months']])

full_observability_by_cohort = rv.observability_status_view(
    config, full_scope, mechanism=mechanism_der_inferred, dimensions=('analysis_cohort',)
)
display(full_observability_by_cohort)

## Stage 10: site and phase explorer

Set `SELECTED_SERIAL` and `SELECTED_PHASE` to inspect a specific site or
phase. Both use aggregated views only -- no per-interval telemetry is ever
loaded into pandas here. Denominator and methodology context are shown next
to each panel so an unassessable site is never mistaken for a
poor-performing one.

In [ ]:
per_site_month_p99_vv = rv.voltvar_denominator_view(
    config, full_scope, mechanism=mechanism_p99_proxy, dimensions=('serial', 'year_utc', 'month_utc')
)
per_site_month_p99_vw = rv.voltwatt_denominator_view(
    config, full_scope, mechanism=mechanism_p99_proxy, dimensions=('serial', 'year_utc', 'month_utc')
)

SELECTED_SERIAL = '810584444'  # SITE_A_SERIAL or SITE_B_SERIAL
print('Selected serial:', SELECTED_SERIAL)

fig = rp.plot_site_profile(per_site_month_p99_vv, per_site_month_p99_vw, SELECTED_SERIAL, context=contexts['p99_net_export_proxy'])
plt.show()

In [ ]:
SELECTED_PHASE = 'A'  # edit to 'A', 'B' or 'C'

observability_by_phase = rv.observability_status_view(config, full_scope, mechanism=mechanism_der_inferred, dimensions=('phase',))
display(observability_by_phase)

observability_metric_by_phase = rv.observability_metric_view(config, full_scope, mechanism=mechanism_der_inferred, dimensions=('phase',))
display(observability_metric_by_phase[observability_metric_by_phase['phase'] == SELECTED_PHASE])

## Stage 11: curtailment status

Only the explicit unavailable-gate-7 panel is rendered. No curtailment
energy, frequency, rate or blended score is computed anywhere in this
notebook.

In [ ]:
rp.plot_curtailment_unavailable()
plt.show()